In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

# 1. Configuración de Chrome
chrome_options = Options()
# chrome_options.add_argument("--headless") # Descomenta esta línea para que no se vea la ventana
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# 2. Inicializar el Driver automáticamente
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

try:
    # 3. URL de búsqueda (Ejemplo: "universal healthcare")
    # Nota: Twitter a veces redirige al login si no estás logueado. 
    # Para pruebas, usaremos una búsqueda que suela ser accesible.
    tema = "universal healthcare"
    url = f"https://twitter.com/search?q={tema}&src=typed_query&f=live"
    
    driver.get(url)
    print(f"Accediendo a: {url}")

    # 4. Esperar a que los tweets aparezcan en el DOM (máximo 10 segundos)
    wait = WebDriverWait(driver, 10)
    
    # X utiliza el atributo 'data-testid="tweetText"' para el cuerpo del mensaje
    wait.until(EC.presence_of_element_located((By.XPATH, '//div[@data-testid="tweetText"]')))

    # 5. Hacer scroll para cargar más contenido
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3) # Pausa para que carguen los nuevos tweets

    # 6. Extraer los textos
    tweets = driver.find_elements(By.XPATH, '//div[@data-testid="tweetText"]')
    
    print(f"\n--- Se encontraron {len(tweets)} tweets ---\n")
    
    data_para_tu_ia = []
    for i, tweet in enumerate(tweets):
        texto = tweet.text.replace('\n', ' ') # Limpiar saltos de línea
        print(f"{i+1}. {texto[:100]}...") # Mostrar los primeros 100 caracteres
        data_para_tu_ia.append(texto)

    # Aquí es donde pasarías 'data_para_tu_ia' a tu capa de análisis de tensión

finally:
    # 7. Cerrar el navegador
    print("\nCerrando navegador...")
    driver.quit()

Accediendo a: https://twitter.com/search?q=universal healthcare&src=typed_query&f=live

Cerrando navegador...


TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x14dcdf3+10b03]
	chromedriver!GetHandleVerifier [0x14dcf24+10c34]
	chromedriver!(No symbol) [0x12c2120]
	chromedriver!(No symbol) [0x130abca]
	chromedriver!(No symbol) [0x130ae6b]
	chromedriver!(No symbol) [0x134d0b2]
	chromedriver!(No symbol) [0x132db54]
	chromedriver!(No symbol) [0x134a9a5]
	chromedriver!(No symbol) [0x132d8a6]
	chromedriver!(No symbol) [0x1300229]
	chromedriver!(No symbol) [0x1300fe4]
	chromedriver!GetHandleVerifier [0x17448b9+2785c9]
	chromedriver!GetHandleVerifier [0x173feb5+273bc5]
	chromedriver!GetHandleVerifier [0x175e06b+291d7b]
	chromedriver!GetHandleVerifier [0x14f5cc8+299d8]
	chromedriver!GetHandleVerifier [0x14fd9fd+3170d]
	chromedriver!GetHandleVerifier [0x14e58c8+195d8]
	chromedriver!GetHandleVerifier [0x14e5a92+197a2]
	chromedriver!GetHandleVerifier [0x14cee9a+2baa]
	KERNEL32!BaseThreadInitThunk [0x750205c9+19]
	ntdll!RtlGetAppContainerNamedObjectPath [0x778c7e4d+ed]
	ntdll!RtlGet

In [2]:
pip install ntscraper


   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -------------------------- ------------- 2.6/4.0 MB 18.9 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 16.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   ------------- -------------------------- 1/3 [lxml]
   -----

In [6]:
from ntscraper import Nitter

scraper = Nitter()

# Manually provide an instance URL (e.g., 'https://nitter.net' or 'https://nitter.poast.org')
tweets = scraper.get_tweets(
    "universal healthcare", 
    mode="term", 
    number=50, 
    instance="https://nitter.net/"
)

data = [tweet["text"] for tweet in tweets["tweets"]]
for text in data:
    print(text[:120])

print(f"\nTotal collected: {len(data)}")

Testing instances: 100%|██████████| 8/8 [00:03<00:00,  2.48it/s]


01-Apr-26 17:17:57 - Empty page on https://nitter.net/

Total collected: 0


In [11]:
import requests
import time

TOPIC = "universal healthcare"
HEADERS = {"User-Agent": "tension-analyzer/1.0"}

subreddits = ["politics", "Conservative", "Liberal", "healthcare", "PoliticalDiscussion"]

data = []

for sub in subreddits:
    url = f"https://www.reddit.com/r/{sub}/search.json?q={TOPIC}&sort=new&limit=10&restrict_sr=1"
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        
        if response.status_code == 200:
            posts = response.json()["data"]["children"]
            for post in posts:
                p = post["data"]
                entry = {
                    "title": p["title"],
                    "body": p.get("selftext", "")[:300],
                    "subreddit": p["subreddit"],
                    "score": p["score"],
                    "num_comments": p["num_comments"],
                }
                data.append(entry)
                print(f"[r/{entry['subreddit']}] {entry['title'][:80]}")
        else:
            print(f"r/{sub} returned status {response.status_code}")
            
    except Exception as e:
        print(f"r/{sub} failed: {e}")
    
    time.sleep(1)  # be polite, avoid rate limiting

print(f"\n✓ Collected {len(data)} posts")

[r/politics] I'm Matt Conroy and I'm a progressive looking to change the way government works
[r/politics] Oregon could become first state to adopt universal healthcare
[r/politics] Big Vote for California Single-Payer Bill Today—What You Need to Know | If passe
[r/politics] Democrats propose California universal healthcare, funded by new income, busines
[r/politics] Democrats propose California universal healthcare, funded by new income, busines
[r/politics] Discussion Thread: President Biden Outlines Plan to Stop the Delta Variant &amp;
[r/politics] I’m Janani Ramachandran, a social justice attorney running for CA State Assembly
[r/politics] How The Phrase “Universal Healthcare” Became Meaningless
[r/politics] Lancet Report: 40% of U.S. COVID Deaths Were Preventable. The Country Needs Univ
[r/politics] Ocasio-Cortez Holds 'Biggest Get Out the Vote Rally of 2020' as Nearly Half a Mi
[r/Conservative] Nancy Pelosi Torn Apart for Telling Republicans 'Hands Off Our Medicaid' Despite
[r/Co

AttributeError: 'list' object has no attribute 'head'

In [14]:
import requests
import json
import time
from datetime import datetime

TOPIC = "universal healthcare"
HEADERS = {"User-Agent": "tension-analyzer/1.0"}

subreddits = ["politics", "Conservative", "Liberal", "healthcare", "PoliticalDiscussion"]

data = []

for sub in subreddits:
    url = f"https://www.reddit.com/r/{sub}/search.json?q={TOPIC}&sort=new&limit=10&restrict_sr=1"
    
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        
        if response.status_code == 200:
            posts = response.json()["data"]["children"]
            for post in posts:
                p = post["data"]
                entry = {
                    "title": p["title"],
                    "body": p.get("selftext", "")[:300],
                    "subreddit": p["subreddit"],
                    "score": p["score"],
                    "num_comments": p["num_comments"],
                    "url": f"https://reddit.com{p['permalink']}",
                    "created_utc": p["created_utc"],
                }
                data.append(entry)
                print(f"[r/{entry['subreddit']}] {entry['title'][:80]}")
        else:
            print(f"r/{sub} returned status {response.status_code}")
            
    except Exception as e:
        print(f"r/{sub} failed: {e}")
    
    time.sleep(1)

print(f"\n✓ Collected {len(data)} posts")

# ── Save to JSON ───────────────────────────────────────
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"reddit_{TOPIC.replace(' ', '_')}_{timestamp}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump({
        "topic": TOPIC,
        "collected_at": timestamp,
        "total_posts": len(data),
        "posts": data
    }, f, ensure_ascii=False, indent=2)

print(f"✓ Saved to {filename}")

[r/politics] I'm Matt Conroy and I'm a progressive looking to change the way government works
[r/politics] Oregon could become first state to adopt universal healthcare
[r/politics] Big Vote for California Single-Payer Bill Today—What You Need to Know | If passe
[r/politics] Democrats propose California universal healthcare, funded by new income, busines
[r/politics] Democrats propose California universal healthcare, funded by new income, busines
[r/politics] Discussion Thread: President Biden Outlines Plan to Stop the Delta Variant &amp;
[r/politics] I’m Janani Ramachandran, a social justice attorney running for CA State Assembly
[r/politics] How The Phrase “Universal Healthcare” Became Meaningless
[r/politics] Lancet Report: 40% of U.S. COVID Deaths Were Preventable. The Country Needs Univ
[r/politics] Ocasio-Cortez Holds 'Biggest Get Out the Vote Rally of 2020' as Nearly Half a Mi
[r/Conservative] Nancy Pelosi Torn Apart for Telling Republicans 'Hands Off Our Medicaid' Despite
[r/Co